# 🎬 → 📝 OpenScribe — video transcription with Whisper large-v3

Give it a video — an upload, a Drive file, or a **link** (Discord CDN attachments included) —
and get an accurate English transcript back as **TXT, SRT, VTT, TSV and JSON**.

**Before you run anything:** set the runtime to a GPU —
`Runtime` → `Change runtime type` → **T4 GPU** (or A100/L4 if you have Colab Pro) → `Save`.

> **A note on TPUs:** Whisper is a PyTorch model and both backends used here
> (`openai-whisper` and `faster-whisper`/CTranslate2) are compiled for **CUDA**, not XLA/TPU.
> A TPU runtime would fall back to the CPU and run *many times slower*, so this notebook
> targets the GPU and uses it as hard as it can (fp16 inference, batching, VAD pre-filtering).
> If you pick a TPU runtime, Cell 1 will tell you.

**Models used** (from the [official Whisper repo](https://github.com/openai/whisper)):

| model | params | VRAM | relative speed | notes |
|---|---|---|---|---|
| `large-v3` | 1550 M | ~10 GB | 1× | **most accurate — the default here** |
| `large-v3-turbo` | 809 M | ~6 GB | ~8× | nearly as good for transcription, much faster |
| `large-v2` | 1550 M | ~10 GB | 1× | slightly less hallucination-prone on some audio |
| `medium.en` / `small.en` | 769 M / 244 M | ~5 / ~2 GB | 2× / 4× | English-only fallbacks |

Just run the cells top to bottom (`Ctrl/Cmd + F9` runs them all).

---
MIT licensed · [github.com/faisal-saddique/openscribe](https://github.com/faisal-saddique/openscribe)

## 1 · Check the runtime

In [ ]:
#@title Run me first — hardware check { display-mode: "form" }
import os, subprocess, sys, textwrap

print("Python:", sys.version.split()[0])

# --- TPU? ---
if os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"):
    print(textwrap.dedent("""
    ⚠️  You are on a TPU runtime.
        Whisper cannot use a TPU (it is CUDA-compiled PyTorch / CTranslate2), so this
        would run on the CPU and take hours. Switch to:
        Runtime → Change runtime type → T4 GPU → Save, then re-run this cell.
    """))

# --- GPU? ---
try:
    smi = subprocess.check_output(["nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader"], text=True).strip()
    GPU_NAME, GPU_MEM, DRIVER = [x.strip() for x in smi.split(",")]
    HAS_GPU = True
    print(f"✅ GPU detected: {GPU_NAME}  ({GPU_MEM}, driver {DRIVER})")
except Exception:
    HAS_GPU, GPU_NAME, GPU_MEM = False, "none", "0 MiB"
    print(textwrap.dedent("""
    ❌ No GPU attached.
       Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save,
       then re-run this cell. (It will still work on CPU, just ~20-40x slower.)
    """))

# ffmpeg is preinstalled on Colab; make sure of it anyway
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    print("Installing ffmpeg…")
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True)
print("✅ ffmpeg:", subprocess.check_output(["ffmpeg", "-version"], text=True).split("\n")[0])

## 2 · Install Whisper

Two backends are installed so you can pick either in Cell 5:

* **`faster-whisper`** — the same Whisper weights re-implemented in CTranslate2. Identical
  accuracy, roughly **4× faster** and uses ~½ the VRAM, plus built-in voice-activity
  detection (VAD) that skips silence and cuts hallucinated text. *Recommended.*
* **`openai-whisper`** — the reference implementation from OpenAI's repo.

Takes ~1–2 minutes. Ignore any pip dependency-resolver warnings.

In [ ]:
#@title Install packages (~1-2 min) { display-mode: "form" }
%pip install -q -U openai-whisper faster-whisper ctranslate2 srt
%pip install -q -U "ffmpeg-python" tqdm yt-dlp requests

import importlib, torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("✅ install complete — if Colab asks you to restart the session, do it and re-run from Cell 1.")

## 3 · Give it a video

Pick a source below and run the cell.

* **url** – paste a direct media link and it downloads server-side. Works with
  **Discord CDN attachments** (`https://cdn.discordapp.com/attachments/...`), S3/GCS links,
  Dropbox, plain file URLs — anything that serves the file directly. If the URL turns out to
  be a *page* rather than a file (YouTube, Vimeo, a Drive share link), it falls back to
  **yt-dlp** and grabs the audio track. **Fastest option — nothing goes through your browser.**
* **upload** – file picker. Fine up to a few hundred MB; browser uploads get flaky above that.
* **drive** – mounts your Google Drive and reads a path from it. Good for big files.
* **path** – a file already sitting in the Colab file browser (`/content/...`).

> **About Discord links:** every `cdn.discordapp.com` URL is signed and **expires about 24 hours**
> after Discord hands it to you — that is what the `?ex=…&is=…&hm=…` query string is. The cell
> decodes `ex` and tells you exactly how long the link has left before it even tries.
> If it has lapsed, right-click the attachment in Discord → *Copy Link* for a fresh one.
> Paste the **entire** URL including everything after the `?`.

Any format ffmpeg can read works: `mp4, mov, mkv, webm, avi, m4a, mp3, wav, ...`

In [ ]:
#@title Load the video { display-mode: "form" }
SOURCE = "url"  #@param ["url", "upload", "drive", "path"]
#@markdown For **url**: paste the full link, including everything after the `?`
VIDEO_URL = ""  #@param {type:"string"}
#@markdown For **drive**: mount happens automatically, then give the path *inside* your Drive,
#@markdown e.g. `MyDrive/videos/lecture.mp4`
DRIVE_PATH = "MyDrive/videos/my_video.mp4"  #@param {type:"string"}
#@markdown For **path**: an absolute path already on this machine
LOCAL_PATH = "/content/my_video.mp4"  #@param {type:"string"}

import os, re, glob, json, shutil, datetime, subprocess, urllib.parse
import requests
from tqdm.auto import tqdm

WORKDIR = "/content/whisper_job"
os.makedirs(WORKDIR, exist_ok=True)
VIDEO_PATH = None
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125 Safari/537.36"


def _signed_link_expiry(url):
    """Discord signs CDN links with ?ex=<hex unix ts>. Returns a UTC datetime or None."""
    q = urllib.parse.parse_qs(urllib.parse.urlparse(url).query)
    for key in ("ex", "Expires", "X-Amz-Expires"):
        if key in q:
            try:
                raw = q[key][0]
                ts = int(raw, 16) if key == "ex" else int(raw)
                return datetime.datetime.utcfromtimestamp(ts)
            except Exception:
                return None
    return None


def _safe_name(name, fallback="download.mp4"):
    name = urllib.parse.unquote(name or "").split("?")[0]
    name = re.sub(r"[^\w.\-]+", "_", name).strip("._")
    return name or fallback


def _ytdlp_fallback(url, dest_dir):
    """The URL is a web page, not a file — let yt-dlp figure it out (audio only: that is all we need)."""
    print("↩️  Not a direct media file — handing it to yt-dlp (audio only)…")
    marker = os.path.join(dest_dir, "_ytdlp_path.txt")
    cmd = ["yt-dlp", "--no-playlist", "-f", "ba/b",
           "-o", os.path.join(dest_dir, "%(title).80s.%(ext)s"),
           "--no-simulate", "--print-to-file", "after_move:filepath", marker,
           "--newline", url]
    try:
        r = subprocess.run(cmd, text=True, capture_output=True)
    except FileNotFoundError:
        raise SystemExit("yt-dlp is not installed — run Cell 2 (Install packages) first.")
    if r.returncode != 0 or not os.path.exists(marker):
        print(r.stderr[-2000:])
        raise RuntimeError("yt-dlp could not fetch that URL.")
    path = open(marker).read().strip().splitlines()[-1]
    os.remove(marker)
    return path


def download_url(url, dest_dir):
    url = url.strip().strip('"').strip("'")
    if not url:
        raise SystemExit("VIDEO_URL is empty — paste a link into the form field.")
    parsed = urllib.parse.urlparse(url)
    host = parsed.netloc

    # -- signed-link expiry check, before we waste a request --
    exp = _signed_link_expiry(url)
    if exp:
        left = exp - datetime.datetime.utcnow()
        if left.total_seconds() <= 0:
            ago = -left.total_seconds()
            raise SystemExit(
                f"⛔ This link expired {int(ago//86400)}d {int(ago%86400//3600)}h ago "
                f"(signature was valid until {exp:%Y-%m-%d %H:%M} UTC).\n"
                "   Discord CDN links last ~24h. Go back to Discord, right-click the "
                "attachment → Copy Link, and paste the fresh URL.")
        print(f"🔑 signed link valid for another "
              f"{int(left.total_seconds())//3600}h {int(left.total_seconds())%3600//60}m "
              f"(until {exp:%Y-%m-%d %H:%M} UTC)")

    name = _safe_name(os.path.basename(parsed.path))
    out = os.path.join(dest_dir, name)

    with requests.get(url, stream=True, timeout=60, allow_redirects=True,
                      headers={"User-Agent": UA, "Accept": "*/*"}) as r:
        if r.status_code in (401, 403, 404) and "discord" in host:
            raise SystemExit(
                f"⛔ Discord returned {r.status_code} for that link.\n"
                "   Either the signature (?ex=…&is=…&hm=…) was truncated when you copied it, "
                "or it has been revoked.\n"
                "   Right-click the attachment in Discord → Copy Link and paste the WHOLE url.")
        r.raise_for_status()

        ctype = r.headers.get("content-type", "").lower()
        if ctype.startswith("text/html"):
            r.close()
            return _ytdlp_fallback(url, dest_dir)

        # give it an extension if the URL had none
        if "." not in os.path.basename(out):
            ext = {"video/mp4": ".mp4", "video/webm": ".webm", "video/quicktime": ".mov",
                   "audio/mpeg": ".mp3", "audio/mp4": ".m4a", "audio/wav": ".wav"}.get(
                       ctype.split(";")[0], ".mp4")
            out += ext

        total = int(r.headers.get("content-length") or 0)
        print(f"⬇️  {host} → {os.path.basename(out)}"
              + (f"  ({total/1e6:,.1f} MB)" if total else ""))
        with open(out, "wb") as f, tqdm(total=total or None, unit="B", unit_scale=True,
                                        unit_divisor=1024, desc="downloading",
                                        dynamic_ncols=True) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk); bar.update(len(chunk))

    if os.path.getsize(out) < 10_000:
        head = open(out, "rb").read(400)
        raise SystemExit(f"⛔ Downloaded only {os.path.getsize(out)} bytes — that is not a media "
                         f"file. First bytes: {head[:200]!r}")
    return out


if SOURCE == "url":
    VIDEO_PATH = download_url(VIDEO_URL, WORKDIR)

elif SOURCE == "upload":
    from google.colab import files
    print("Choose your video file…")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file chosen.")
    name = list(uploaded.keys())[0]
    VIDEO_PATH = os.path.join(WORKDIR, name)
    shutil.move(name, VIDEO_PATH)

elif SOURCE == "drive":
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    VIDEO_PATH = os.path.join("/content/drive", DRIVE_PATH.lstrip("/"))

else:
    VIDEO_PATH = LOCAL_PATH

if not os.path.isfile(VIDEO_PATH):
    raise FileNotFoundError(f"Can't find: {VIDEO_PATH}")

# --- probe it ---
probe = json.loads(subprocess.check_output([
    "ffprobe", "-v", "quiet", "-print_format", "json",
    "-show_format", "-show_streams", VIDEO_PATH], text=True))
DURATION = float(probe["format"].get("duration", 0))
size_mb = os.path.getsize(VIDEO_PATH) / 1e6
has_audio = any(s["codec_type"] == "audio" for s in probe["streams"])

print(f"\n📼 {os.path.basename(VIDEO_PATH)}")
print(f"   {size_mb:,.1f} MB | {DURATION/60:.1f} min | audio track: {'yes' if has_audio else 'NO ❌'}")
if not has_audio:
    raise SystemExit("This file has no audio stream — nothing to transcribe.")

## 4 · Extract the audio

Whisper wants 16 kHz mono. Doing this once up front is faster and more reliable than letting
each backend decode the video itself.

In [ ]:
#@title Extract 16 kHz mono WAV { display-mode: "form" }
#@markdown Light denoise + loudness normalisation. Helps on noisy or quiet recordings,
#@markdown leave it off for clean studio audio.
CLEAN_AUDIO = False  #@param {type:"boolean"}

import os, subprocess, time

AUDIO_PATH = os.path.join(WORKDIR, "audio_16k.wav")
filters = "highpass=f=60,afftdn=nf=-25,loudnorm=I=-16:TP=-1.5:LRA=11" if CLEAN_AUDIO else "loudnorm=I=-16:TP=-1.5:LRA=11"

t0 = time.time()
cmd = ["ffmpeg", "-y", "-i", VIDEO_PATH, "-vn", "-sn", "-dn",
       "-af", filters, "-ac", "1", "-ar", "16000",
       "-c:a", "pcm_s16le", AUDIO_PATH]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print(res.stderr[-3000:])
    raise RuntimeError("ffmpeg failed")

print(f"✅ audio extracted in {time.time()-t0:.1f}s → {os.path.getsize(AUDIO_PATH)/1e6:.1f} MB")

## 5 · Settings

Defaults are tuned for **best-quality English transcription on a Colab T4**.

In [ ]:
#@title Transcription settings { display-mode: "form" }
BACKEND = "faster-whisper"  #@param ["faster-whisper", "openai-whisper"]
MODEL   = "large-v3"        #@param ["large-v3", "large-v3-turbo", "large-v2", "medium.en", "small.en"]
LANGUAGE = "en"             #@param ["en", "auto"]
TASK     = "transcribe"     #@param ["transcribe", "translate"]

#@markdown ---
#@markdown **Quality knobs**
BEAM_SIZE = 5  #@param {type:"slider", min:1, max:10, step:1}
#@markdown Voice-activity detection: skips silence, the single best defence against Whisper
#@markdown inventing text over quiet passages. (`faster-whisper` only.)
VAD_FILTER = True  #@param {type:"boolean"}
#@markdown Per-word timestamps — needed for karaoke-style subtitles, ~15% slower.
WORD_TIMESTAMPS = True  #@param {type:"boolean"}
#@markdown Feed previous text as context: better coherence, but can propagate a mistake.
CONDITION_ON_PREVIOUS = False  #@param {type:"boolean"}
#@markdown `float16` = fastest on any modern GPU. Use `int8_float16` only if you run out of VRAM.
COMPUTE_TYPE = "float16"  #@param ["float16", "int8_float16", "int8"]

#@markdown ---
#@markdown **Subtitle formatting**
MAX_LINE_WIDTH = 42  #@param {type:"slider", min:30, max:90, step:1}
MAX_LINE_COUNT = 2   #@param {type:"slider", min:1, max:3, step:1}

#@markdown ---
#@markdown Nudge spelling of names/jargon, e.g. `Anthropic, Faisal, Kubernetes, Karachi`
INITIAL_PROMPT = ""  #@param {type:"string"}

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    COMPUTE_TYPE = "int8"
    print("⚠️  Running on CPU — expect this to be very slow. Consider a GPU runtime.")

# turbo naming differs between the two backends
_MODEL_MAP = {
    "openai-whisper": {"large-v3-turbo": "turbo"},
    "faster-whisper": {},
}
MODEL_ID = _MODEL_MAP[BACKEND].get(MODEL, MODEL)

print(f"backend={BACKEND}  model={MODEL_ID}  device={DEVICE}  compute={COMPUTE_TYPE}")
print(f"language={LANGUAGE}  task={TASK}  beam={BEAM_SIZE}  vad={VAD_FILTER}  words={WORD_TIMESTAMPS}")

## 6 · Transcribe

First run downloads the model (~1.5 GB for `large-v3`), so give it a minute.
On a T4 with `faster-whisper` + `large-v3` expect roughly **8–12× real time** — a 60-minute
video lands in about 5–8 minutes.

In [ ]:
#@title Run the transcription { display-mode: "form" }
import time, gc, torch
from tqdm.auto import tqdm

SEGMENTS = []      # normalised: {start, end, text, words:[{start,end,word,probability}]}
INFO = {}
t0 = time.time()


def _run_faster_whisper():
    from faster_whisper import WhisperModel
    model = WhisperModel(MODEL_ID, device=DEVICE, compute_type=COMPUTE_TYPE,
                         download_root="/content/whisper_models")
    seg_iter, info = model.transcribe(
        AUDIO_PATH,
        language=None if LANGUAGE == "auto" else LANGUAGE,
        task=TASK,
        beam_size=BEAM_SIZE,
        best_of=BEAM_SIZE,
        patience=1.0,
        temperature=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        compression_ratio_threshold=2.4,
        log_prob_threshold=-1.0,
        no_speech_threshold=0.6,
        condition_on_previous_text=CONDITION_ON_PREVIOUS,
        word_timestamps=WORD_TIMESTAMPS,
        vad_filter=VAD_FILTER,
        vad_parameters=dict(min_silence_duration_ms=500, speech_pad_ms=200),
        initial_prompt=INITIAL_PROMPT or None,
    )
    out = []
    bar = tqdm(total=round(info.duration, 2), unit="s", desc="transcribing", dynamic_ncols=True)
    done = 0.0
    for s in seg_iter:
        out.append({
            "start": s.start, "end": s.end, "text": s.text.strip(),
            "words": [{"start": w.start, "end": w.end, "word": w.word,
                       "probability": w.probability} for w in (s.words or [])],
            "avg_logprob": s.avg_logprob, "no_speech_prob": s.no_speech_prob,
        })
        bar.update(max(0.0, s.end - done)); done = s.end
    bar.update(max(0.0, round(info.duration, 2) - done)); bar.close()
    meta = {"language": info.language, "language_probability": info.language_probability,
            "duration": info.duration}
    del model; gc.collect(); torch.cuda.empty_cache()
    return out, meta


def _run_openai_whisper():
    import whisper
    model = whisper.load_model(MODEL_ID, device=DEVICE, download_root="/content/whisper_models")
    result = model.transcribe(
        AUDIO_PATH,
        language=None if LANGUAGE == "auto" else LANGUAGE,
        task=TASK,
        beam_size=BEAM_SIZE,
        best_of=BEAM_SIZE,
        temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        condition_on_previous_text=CONDITION_ON_PREVIOUS,
        word_timestamps=WORD_TIMESTAMPS,
        initial_prompt=INITIAL_PROMPT or None,
        fp16=(DEVICE == "cuda"),
        verbose=False,
    )
    out = []
    for s in result["segments"]:
        out.append({
            "start": s["start"], "end": s["end"], "text": s["text"].strip(),
            "words": [{"start": w["start"], "end": w["end"],
                       "word": w["word"], "probability": w.get("probability")}
                      for w in s.get("words", [])],
            "avg_logprob": s.get("avg_logprob"), "no_speech_prob": s.get("no_speech_prob"),
        })
    meta = {"language": result.get("language"), "language_probability": None,
            "duration": DURATION}
    del model; gc.collect(); torch.cuda.empty_cache()
    return out, meta


try:
    SEGMENTS, INFO = (_run_faster_whisper() if BACKEND == "faster-whisper"
                      else _run_openai_whisper())
except Exception as e:
    # The usual culprit is a cuDNN/cuBLAS mismatch for CTranslate2 on a fresh Colab image.
    print(f"\n⚠️  {BACKEND} failed: {type(e).__name__}: {e}")
    if BACKEND == "faster-whisper":
        print("↩️  Falling back to the reference openai-whisper backend…")
        MODEL_ID = {"large-v3-turbo": "turbo"}.get(MODEL, MODEL)
        BACKEND = "openai-whisper"
        SEGMENTS, INFO = _run_openai_whisper()
    else:
        raise

elapsed = time.time() - t0
audio_len = INFO.get("duration") or DURATION
print(f"\n✅ done in {elapsed/60:.1f} min  ({audio_len/max(elapsed,1e-9):.1f}× real time)")
print(f"   {len(SEGMENTS)} segments | detected language: {INFO.get('language')}"
      + (f" ({INFO['language_probability']:.0%} confident)" if INFO.get("language_probability") else ""))

## 7 · Write the output files

`.txt` (plain transcript), `.srt` + `.vtt` (subtitles), `.tsv` (timestamped table),
`.json` (everything, incl. word timings and confidence).

In [ ]:
#@title Save TXT / SRT / VTT / TSV / JSON { display-mode: "form" }
import os, json, textwrap, re

BASE = re.sub(r"[^\w.\-]+", "_", os.path.splitext(os.path.basename(VIDEO_PATH))[0]).strip("_") or "transcript"
OUTDIR = os.path.join(WORKDIR, "transcript")
os.makedirs(OUTDIR, exist_ok=True)


def ts(seconds, sep=","):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"


def wrap(text):
    lines = textwrap.wrap(text, width=MAX_LINE_WIDTH) or [text]
    if len(lines) > MAX_LINE_COUNT:      # keep cue height sane
        head = lines[:MAX_LINE_COUNT - 1]
        head.append(" ".join(lines[MAX_LINE_COUNT - 1:]))
        lines = head
    return "\n".join(lines)


paths = {k: os.path.join(OUTDIR, f"{BASE}.{k}") for k in ("txt", "srt", "vtt", "tsv", "json")}

# plain text — paragraph break where there is a pause > 2s
paras, cur, prev_end = [], [], None
for s in SEGMENTS:
    if prev_end is not None and s["start"] - prev_end > 2.0 and cur:
        paras.append(" ".join(cur)); cur = []
    cur.append(s["text"]); prev_end = s["end"]
if cur:
    paras.append(" ".join(cur))
full_text = "\n\n".join(re.sub(r"\s+", " ", p).strip() for p in paras)
open(paths["txt"], "w", encoding="utf-8").write(full_text + "\n")

# srt
with open(paths["srt"], "w", encoding="utf-8") as f:
    for i, s in enumerate(SEGMENTS, 1):
        f.write(f"{i}\n{ts(s['start'])} --> {ts(s['end'])}\n{wrap(s['text'])}\n\n")

# vtt
with open(paths["vtt"], "w", encoding="utf-8") as f:
    f.write("WEBVTT\n\n")
    for s in SEGMENTS:
        f.write(f"{ts(s['start'], '.')} --> {ts(s['end'], '.')}\n{wrap(s['text'])}\n\n")

# tsv
with open(paths["tsv"], "w", encoding="utf-8") as f:
    f.write("start_ms\tend_ms\tstart_hms\ttext\n")
    for s in SEGMENTS:
        f.write(f"{int(s['start']*1000)}\t{int(s['end']*1000)}\t{ts(s['start'])}\t{s['text']}\n")

# json
with open(paths["json"], "w", encoding="utf-8") as f:
    json.dump({
        "source_file": os.path.basename(VIDEO_PATH),
        "backend": BACKEND, "model": MODEL_ID, "task": TASK,
        "language": INFO.get("language"),
        "duration_sec": INFO.get("duration") or DURATION,
        "settings": {"beam_size": BEAM_SIZE, "vad_filter": VAD_FILTER,
                     "word_timestamps": WORD_TIMESTAMPS,
                     "condition_on_previous_text": CONDITION_ON_PREVIOUS,
                     "compute_type": COMPUTE_TYPE, "initial_prompt": INITIAL_PROMPT},
        "text": full_text, "segments": SEGMENTS,
    }, f, ensure_ascii=False, indent=2)

words = len(full_text.split())
print(f"📝 {words:,} words, {len(SEGMENTS)} segments\n")
for k, p in paths.items():
    print(f"   {k:>4} · {os.path.getsize(p)/1024:8.1f} KB · {p}")

## 8 · Preview

In [ ]:
#@title Preview the transcript { display-mode: "form" }
PREVIEW_CHARS = 3000  #@param {type:"slider", min:500, max:20000, step:500}
SHOW_TIMESTAMPS = True  #@param {type:"boolean"}

if SHOW_TIMESTAMPS:
    shown = 0
    for s in SEGMENTS:
        line = f"[{ts(s['start'])[:-4]} → {ts(s['end'])[:-4]}]  {s['text']}"
        print(line); shown += len(line)
        if shown > PREVIEW_CHARS:
            print(f"\n… ({len(SEGMENTS)} segments total — open the .txt for the rest)"); break
else:
    print(full_text[:PREVIEW_CHARS])
    if len(full_text) > PREVIEW_CHARS:
        print(f"\n… ({len(full_text):,} characters total)")

## 9 · Quick quality check

Whisper's failure mode is *fluent nonsense* rather than obvious garbage, so it is worth a
glance at the low-confidence segments before you trust the transcript.

In [ ]:
#@title Flag suspicious segments { display-mode: "form" }
import collections

low_conf, repeats = [], []
counts = collections.Counter(s["text"].strip().lower() for s in SEGMENTS if s["text"].strip())

for s in SEGMENTS:
    lp = s.get("avg_logprob")
    if lp is not None and lp < -1.0:
        low_conf.append(s)
    if counts[s["text"].strip().lower()] >= 4 and len(s["text"]) > 12:
        repeats.append(s)

print(f"segments: {len(SEGMENTS)}")
print(f"low confidence (avg_logprob < -1.0): {len(low_conf)}")
print(f"lines repeated 4+ times (possible loop/hallucination): {len(set(r['text'] for r in repeats))}\n")

for s in low_conf[:10]:
    print(f"  ⚠️  [{ts(s['start'])[:-4]}] ({s['avg_logprob']:.2f})  {s['text'][:100]}")
for t in list(dict.fromkeys(r["text"] for r in repeats))[:5]:
    print(f"  🔁  x{counts[t.strip().lower()]}  {t[:100]}")

if not low_conf and not repeats:
    print("  ✅ nothing suspicious — the transcript looks clean.")
else:
    print("\n  Tip: turn VAD_FILTER on, set CONDITION_ON_PREVIOUS off, or enable CLEAN_AUDIO"
          "\n  in Cell 4, then re-run Cells 4-7.")

## 10 · Download / save to Drive

In [ ]:
#@title Get the files { display-mode: "form" }
DOWNLOAD_ZIP = True   #@param {type:"boolean"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown Folder inside your Drive to copy the transcript into
DRIVE_OUT_DIR = "MyDrive/transcripts"  #@param {type:"string"}

import shutil, os

zip_base = os.path.join(WORKDIR, f"{BASE}_transcript")
zip_path = shutil.make_archive(zip_base, "zip", OUTDIR)
print(f"📦 {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)")

if SAVE_TO_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    dest = os.path.join("/content/drive", DRIVE_OUT_DIR.lstrip("/"), BASE)
    os.makedirs(dest, exist_ok=True)
    for f in os.listdir(OUTDIR):
        shutil.copy2(os.path.join(OUTDIR, f), dest)
    print(f"💾 copied to Drive → {dest}")

if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(zip_path)

---

## Troubleshooting & tuning

**"CUDA out of memory"** — set `COMPUTE_TYPE = "int8_float16"` in Cell 5, or drop to
`large-v3-turbo`. Then `Runtime → Restart session` and re-run from Cell 5.

**`faster-whisper` errors about `libcudnn` / `libcublas`** — Cell 6 already catches this and
falls back to `openai-whisper` automatically. To fix it properly instead, run
`!pip install -q "ctranslate2==4.4.0"` and restart the session.

**Transcript loops or invents sentences over music/silence** — keep `VAD_FILTER` on, set
`CONDITION_ON_PREVIOUS = False`, and turn on `CLEAN_AUDIO` in Cell 4.

**Names and jargon spelled wrong** — list them in `INITIAL_PROMPT` (Cell 5). Whisper treats it
as preceding context and biases spelling towards it.

**Subtitles too long on screen** — lower `MAX_LINE_WIDTH` (Cell 5) and re-run Cell 7 only;
no need to transcribe again.

**Speaker labels ("who said what")** — Whisper does not do diarisation. Add
[`pyannote.audio`](https://github.com/pyannote/pyannote-audio) (needs a free Hugging Face token)
and align its speaker turns against the word timestamps already in the `.json`.

**A Discord link that worked yesterday now 403s** — the signature in `?ex=…&is=…&hm=…`
lasts about 24 hours. Copy the link again from Discord; nothing else needs changing.

**Colab disconnected mid-run** — long jobs on the free tier get reclaimed. Use the `drive`
source so the video survives, and prefer `large-v3-turbo` for anything over ~2 hours.

**Reference:** [github.com/openai/whisper](https://github.com/openai/whisper) ·
[SYSTRAN/faster-whisper](https://github.com/SYSTRAN/faster-whisper)